# Introducción a xarray

Este es un notebook utilizado como introducción al paquete de python [xarray](https://docs.xarray.dev/en/stable/):

> Xarray introduce etiquetas en la forma de dimensiones, coordenadas y atributos en matrices sin procesar similar a las de Numpy, lo que permite una experiencia de desarrollo más intuitiva, más concisa y menos propensa a errores. El paquete incluye una gran librería en crecimiento, de funciones independientes de dominio (domain-agnostic) para analisis avanzados y visualización con estas estructuras de datos.
> Xarray fue inspirado por y se basa fuertemente en pandas, el popular paquete de análisis de datos enfocado en datos tabulares etiquetados. Está particularmente adaptado para trabajar con netCDF, los cuáles fueron la fuente de modelos de datos de Xarray, y se integra estrechamente con dask para computación paralela.

Estos notebook son una reinterpretación en español de la serie "introduction_to_xarray" del usuario [
coecms-training ](https://github.com/coecms-training/introduction_to_xarray/tree/master). Cuyo formato opcional del turorial se encuentra en el canal de youtube [CLEX CMS youtube channel](https://www.youtube.com/channel/UCSmoK6oWV9O0Hmyt9UdDNsQ)

Para los datos se utilizo, la base da datos Madrigal, el cuál contempla datos ionosféricos en formato netCDF4. Utilizandose los datos específicos del día de tormeta geomagnética del 16-18 de mayo del 2025, obtenidos del Radio Observatorio de Jicamarca. El fin no es hacer una análisis del evento, sino hacer una breve exploración en función de las herramientas de xarray.

Esta serie consiste de notebooks de igual manera a [aidanheerdegen](https://github.com/coecms-training/introduction_to_xarray/tree/master) divide los temas en:

1. Lectura de datos y metadatos de un archivo netCDF dentro de un dataset xarray
2. Separación de un dataset por tiempo y espacio
3. Ploteo
4. Cálculo de métricas (e.g. promedio, máximo)
5. Masking
6. Apertura de multiples archivos como un solo dataset
7. Guardado de datasets a un netCDF

# Lectura de un dataset
Un dataset [xarray](https://docs.xarray.dev/en/stable/user-guide/data-structures.html#dataset) es un contenedor para datos y su metadata asociada, incluyendo coordenadas etiquetadas

Primer paso, importar el paquete xarray

In [1]:
import xarray

Cuando abrimos un archivo netCDF, la metadata del archivo es leído y almacenado como un ```xarray.Dataset```. En este caso, el archivo es accedido via un servidor [OpenDap](https://www.opendap.org/) ya que es universalmente accesible. El comando equivalente si el archivo netCDF fue guardado en el mismo directorio como un notebook es mostrado como comentario como referencia

In [2]:
ds = xarray.open_dataset('jro20250516_132000.nc')

In [3]:
ds

<xarray.Dataset> Size: 323kB
Dimensions:     (timestamps: 154, gdalt: 43)
Coordinates:
  * timestamps  (timestamps) float64 1kB 1.747e+09 1.747e+09 ... 1.747e+09
  * gdalt       (gdalt) float64 344B 180.0 195.0 210.0 ... 780.0 795.0 810.0
Data variables:
    gdlatr      (timestamps) float64 1kB ...
    gdlonr      (timestamps) float64 1kB ...
    inttms      (timestamps) float64 1kB ...
    dne         (timestamps, gdalt) float64 53kB ...
    dte         (timestamps, gdalt) float64 53kB ...
    dti         (timestamps, gdalt) float64 53kB ...
    ne          (timestamps, gdalt) float64 53kB ...
    te          (timestamps, gdalt) float64 53kB ...
    ti          (timestamps, gdalt) float64 53kB ...
Attributes: (12/16)
    catalog_text:          Catalog information from record 0:                ...
    header_text:           Header information from record 1:                 ...
    instrument:            Jicamarca IS Radar
    instrument_code:       10
    kind_of_data_file:     Faraday rotation with alternating code Long Pulse
    kindat_code:           1800
    ...                    ...
    instrument_latitude:   -11.95
    instrument_longitude:  283.13
    instrument_altitude:   0.525
    instrument_category:   Incoherent Scatter Radars
    instrument_PI:         Danny Scipion
    instrument_PI_email:   dscipion@igp.gob.pe

Hay cuatro secciones a notar: ``Dimensions``,``Coordinates``,``Data Variables``,``Attributes``

``Dimensions`` da el tamaño de cada dimensión mencionada. Esto es un dataset compatible con [CF](http://cfconventions.org), lo que significa que cualquier variable que tenga el mismo nombre como una dimension es una coordenada. Existen otros metadatos que pueden utilizarse para indicar una coordenada. En este caso, ``xarray`` hace referencia a las coordenadas asociadas a las variables con ``*``.

# Accediendo a los datos

El comando ``open_dataset`` solo lee la metadata de los archivos netCDF. No intenta leer cualquier dato hasta que hay una operación que lo requiera.

EL objeto ``xarray.DataSet`` tiene un número de métodos para acceder a las coordenadas, atributos y datos. Las variable de los datos son guardados en una estructura como ``dict``, ``ds.data_vars``:

In [4]:
ds.data_vars

Data variables:
    gdlatr   (timestamps) float64 1kB ...
    gdlonr   (timestamps) float64 1kB ...
    inttms   (timestamps) float64 1kB ...
    dne      (timestamps, gdalt) float64 53kB ...
    dte      (timestamps, gdalt) float64 53kB ...
    dti      (timestamps, gdalt) float64 53kB ...
    ne       (timestamps, gdalt) float64 53kB ...
    te       (timestamps, gdalt) float64 53kB ...
    ti       (timestamps, gdalt) float64 53kB ...

Es posible hacer un bucle de los variables de los datos del dataset, lo cuál retorna cada nombra de variable:

In [5]:
for varname in ds:
    print(varname)

gdlatr
gdlonr
inttms
dne
dte
dti
ne
te
ti


Una variable individual puede ser accedida usando su nombre, ya sea como un ``dict`` tipo clave

In [6]:
ds['ne']

<xarray.DataArray 'ne' (timestamps: 154, gdalt: 43)> Size: 53kB
[6622 values with dtype=float64]
Coordinates:
  * timestamps  (timestamps) float64 1kB 1.747e+09 1.747e+09 ... 1.747e+09
  * gdalt       (gdalt) float64 344B 180.0 195.0 210.0 ... 780.0 795.0 810.0
Attributes:
    units:        m-3
    description:  Electron density (Ne)

or using tha variable name as python attribute

In [7]:
ds.ne

<xarray.DataArray 'ne' (timestamps: 154, gdalt: 43)> Size: 53kB
[6622 values with dtype=float64]
Coordinates:
  * timestamps  (timestamps) float64 1kB 1.747e+09 1.747e+09 ... 1.747e+09
  * gdalt       (gdalt) float64 344B 180.0 195.0 210.0 ... 780.0 795.0 810.0
Attributes:
    units:        m-3
    description:  Electron density (Ne)

Así ``ds.tas`` es un ``xarray.DataArray`` y tiene su propia metadata dando más información acerca de la variable misma. En este caso es la densidad de electrones